**PARTE 0: DATOS**
**Paso 1: Preparación del entorno y carga de datos**

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime
url_viajes = "yellow_tripdata_2024-01.parquet"
url_zonas = "taxi_zone_lookup.csv"

columnas_a_cargar = [
    "tpep_pickup_datetime", 
    "PULocationID", 
    "trip_distance", 
    "fare_amount",
    "tip_amount"
]
viajes = pd.read_parquet(url_viajes, columns=columnas_a_cargar)
zonas = pd.read_csv(url_zonas)

**Paso 2: Evaluar lo cargado**

In [2]:
viajes.head()

,tpep_pickup_datetime,PULocationID,trip_distance,fare_amount,tip_amount
0,2024-01-01 00:57:55,186,1.72,17.7,0.00
1,2024-01-01 00:03:00,140,1.80,10.0,3.75
2,2024-01-01 00:17:06,236,4.70,23.3,3.00
3,2024-01-01 00:36:38,79,1.40,10.0,2.00
4,2024-01-01 00:46:51,211,0.80,7.9,3.20


In [3]:
fecha_max = viajes["tpep_pickup_datetime"].max()
fecha_min = viajes["tpep_pickup_datetime"].min()

print(f"La fecha mínima:{fecha_min}, y la fecha máxima:{fecha_max}")

La fecha mínima:2002-12-31 22:59:39, y la fecha máxima:2024-02-01 00:01:15


In [4]:
viajes.dtypes

tpep_pickup_datetime    datetime64[us]
PULocationID                     int32
trip_distance                  float64
fare_amount                    float64
tip_amount                     float64
dtype: object

In [5]:
viajes.shape

(2964624, 5)

In [6]:
zonas.head()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


**Paso 3: Filtrar a Enero de 2024**

In [8]:
viajes_enero = viajes[(viajes["tpep_pickup_datetime"]>="2024-01-01")&(viajes["tpep_pickup_datetime"]<"2024-02-01")]

fecha_max_en = viajes_enero["tpep_pickup_datetime"].max()
fecha_min_en = viajes_enero["tpep_pickup_datetime"].min()

print(f"La fecha mínima:{fecha_min_en}, y la fecha máxima:{fecha_max_en}")

La fecha mínima:2024-01-01 00:00:00, y la fecha máxima:2024-01-31 23:59:55


**Paso 4: Filtrar zonas**

In [13]:
# 1. Viajes por zona de origen (solo enero)
conteo = viajes_enero.groupby("PULocationID").size().reset_index(name="viajes")

# 2. Cruzar con el catálogo de zonas (los nombres de la columna difieren)
zonas_filtradas = conteo.merge(zonas, left_on="PULocationID", right_on="LocationID", how="left")

# 3. Filtrar: 3,000 viajes o más y sin Unknown
zonas_pronostico = zonas_filtradas[
    (zonas_filtradas["viajes"] >= 3000)
    & (zonas_filtradas["Zone"] != "Unknown")
    & (zonas_filtradas["Borough"] != "Unknown")
]

print(len(zonas_pronostico))

58


In [12]:
zonas[(zonas["Zone"] == "Unknown") | (zonas["Borough"] == "Unknown")]

,LocationID,Borough,Zone,service_zone
263,264,Unknown,NaN,NaN


In [15]:
zonas_filtradas.head()

,PULocationID,viajes,LocationID,Borough,Zone,service_zone
0,1,295,1,EWR,Newark Airport,EWR
1,2,3,2,Queens,Jamaica Bay,Boro Zone
2,3,105,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,3568,4,Manhattan,Alphabet City,Yellow Zone
4,6,21,6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone


**Paso 5: Viajes a serie de tiempo**

In [16]:
# 1. Solo viajes de las zonas pronosticables
ids = sorted(zonas_pronostico["PULocationID"])
v = viajes_enero[viajes_enero["PULocationID"].isin(ids)].copy()

# 2. Redondear la hora de inicio hacia abajo y contar viajes por zona-hora
v["hora"] = v["tpep_pickup_datetime"].dt.floor("h")
conteo = v.groupby(["PULocationID", "hora"]).size().rename("viajes")

# 3. Todas las combinaciones zona-hora (1 ene a 1 feb), con 0 donde no hubo viajes
horas = pd.date_range("2024-01-01 00:00", "2024-02-01 23:00", freq="h")
indice = pd.MultiIndex.from_product([ids, horas], names=["PULocationID", "hora"])
demanda = conteo.reindex(indice, fill_value=0).reset_index()

# 4. El 1 de febrero es el "mañana": no hay datos reales, así que va vacío (no 0)
demanda.loc[demanda["hora"] >= "2024-02-01", "viajes"] = np.nan

In [17]:
print(demanda.shape)
print(demanda["viajes"].sum(), len(v))
print(demanda["viajes"].isna().sum())

(44544, 3)
2885953.0 2885953
1392


In [18]:
demanda.head()

,PULocationID,hora,viajes
0,4,2024-01-01 00:00:00,25.0
1,4,2024-01-01 01:00:00,29.0
2,4,2024-01-01 02:00:00,34.0
3,4,2024-01-01 03:00:00,31.0
4,4,2024-01-01 04:00:00,32.0


**PARTE 1: PRONOSTICO**
**Las variables lags y calendario**

In [21]:
# 1. Ordenar: el shift depende del orden
demanda = demanda.sort_values(by=["PULocationID", "hora"]).reset_index(drop=True)

# 2. Lags: viajes pasados DE LA MISMA ZONA
for k in [24, 48, 167, 168, 169]:
    demanda[f"lag_{k}"] = demanda.groupby(["PULocationID"])["viajes"].shift(k)

# 3. Promedio del día anterior (media de viajes por hora del día anterior)
#    Pista: ¿cómo sacas la media de las últimas 24 horas, sin incluir las horas del día actual?
demanda["media_dia_ant"] = demanda.groupby("PULocationID")["viajes"].transform(lambda x: x.shift(24).rolling(window=24, min_periods=24).mean())

# 4. Calendario (pista: el accesor .dt de la columna "hora")
demanda["hora_dia"] = demanda["hora"].dt.hour
demanda["dia_semana"] = demanda["hora"].dt.day_of_week
demanda["feriado"] = (demanda["hora"].dt.date == datetime.date(2024, 1, 15)).astype(int) # 1 si es el 15 de enero de 2024, 0 si no

# 5. La zona como categoría (para XGBoost)
demanda["zona"] = demanda["PULocationID"].astype("category")

In [22]:
print(demanda["feriado"].sum())
print(demanda["zona"].dtype, demanda["zona"].nunique())
demanda[demanda["PULocationID"] == demanda["PULocationID"].iloc[0]].iloc[[0, 24, 168, 200]][["hora", "viajes", "lag_24", "lag_168", "media_dia_ant"]]

1392
category 58


,hora,viajes,lag_24,lag_168,media_dia_ant
0,2024-01-01 00:00:00,25.0,NaN,NaN,NaN
24,2024-01-02 00:00:00,1.0,25.0,NaN,NaN
168,2024-01-08 00:00:00,1.0,30.0,25.0,6.500000
200,2024-01-09 08:00:00,3.0,3.0,6.0,1.291667


**Separación entrenamiento, prueba y futuro**
VARIABLES = ["lag_24", ..., "media_dia_ant", "hora_dia", ..., "zona"]

entrenamiento = demanda[ (condición sobre "hora") ]
prueba        = demanda[ (condición 1) & (condición 2) ]
futuro        = demanda[ (condición sobre "hora") ]

# Pista: ¿qué filas de entrenamiento tienen NaN en lag_168 y no sirven para aprender?